# Event Study & Visualization
MACS 30113 Final Project — Analyzing and Modeling part, Anyi Li

Reads sentiment-scored data from Notebook 1. Computes monthly aggregates (average sentiment, posting volume, volatility) across subreddits, runs statistical tests comparing pre/post COVID periods, and saves all results to S3 as CSVs. **Plots are created locally** — see `4_visualizations_local.py`.

In [1]:
%%configure -f
{
    "conf": {
        "spark.pyspark.python": "python3",
        "spark.pyspark.virtualenv.enabled": "true",
        "spark.pyspark.virtualenv.type": "native",
        "spark.pyspark.virtualenv.bin.path": "/usr/bin/virtualenv"
    }
}

In [2]:
spark

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,Current session?
7,application_1779939149554_0009,pyspark,idle,Link,Link,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

## 1. Load Sentiment-Scored Data
Reads the output of Notebook 1. Run Notebook 1 first if this path does not exist.

In [3]:
reddit_df = spark.read.parquet("s3://30113-final-project/results/sentiment_scored/")

print("Total records:", reddit_df.count())
reddit_df.select("subreddit", "period", "vader_sentiment").show(5)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Total records: 4538
+----------+----------+---------------+
| subreddit|    period|vader_sentiment|
+----------+----------+---------------+
|depression|post_covid|         -0.961|
|depression|post_covid|        -0.5251|
|depression|post_covid|         0.9337|
|depression|post_covid|         0.9468|
|depression|post_covid|         0.6124|
+----------+----------+---------------+
only showing top 5 rows

## 2. Monthly Aggregations
Computes three metrics per subreddit per month, all running distributed in Spark:
- `avg_sentiment`: mean VADER compound score
- `sentiment_volatility`: standard deviation of VADER scores (measures emotional instability)
- `post_volume`: total number of posts and comments

In [4]:
from pyspark.sql.functions import avg, stddev, count, col, round as spark_round

monthly_stats = reddit_df.groupBy("subreddit", "year", "month", "period").agg(
    spark_round(avg("vader_sentiment"), 4).alias("avg_sentiment"),
    spark_round(stddev("vader_sentiment"), 4).alias("sentiment_volatility"),
    count("*").alias("post_volume")
).orderBy("subreddit", "year", "month")

monthly_stats.show(20)
print("Row count:", monthly_stats.count())

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+------------+----+-----+----------+-------------+--------------------+-----------+
|   subreddit|year|month|    period|avg_sentiment|sentiment_volatility|post_volume|
+------------+----+-----+----------+-------------+--------------------+-----------+
|     anxiety|2020|    3|post_covid|       0.0776|              0.6155|       1680|
|  depression|2020|    3|post_covid|       0.0387|              0.6412|       1935|
|mentalhealth|2020|    3|post_covid|       0.0771|              0.6484|        923|
+------------+----+-----+----------+-------------+--------------------+-----------+

Row count: 3

In [5]:
# Period-level summary: average metrics per subreddit per period
period_comparison = reddit_df.groupBy("subreddit", "period").agg(
    spark_round(avg("vader_sentiment"), 4).alias("avg_sentiment"),
    spark_round(stddev("vader_sentiment"), 4).alias("sentiment_volatility"),
    count("*").alias("post_volume")
).orderBy("subreddit", "period")

period_comparison.show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+------------+----------+-------------+--------------------+-----------+
|   subreddit|    period|avg_sentiment|sentiment_volatility|post_volume|
+------------+----------+-------------+--------------------+-----------+
|     anxiety|post_covid|       0.0776|              0.6155|       1680|
|  depression|post_covid|       0.0387|              0.6412|       1935|
|mentalhealth|post_covid|       0.0771|              0.6484|        923|
+------------+----------+-------------+--------------------+-----------+

## 3. Statistical Tests (t-test)
Two-sample t-test comparing pre vs. post COVID VADER sentiment scores per subreddit. Uses a small sample collected to the driver to avoid memory issues.

**Note:** Pre-COVID data may be empty if only the post-COVID test batch is loaded.

In [9]:
import math

results = []

for subreddit_name in ["depression", "anxiety", "mentalhealth"]:
    pre_stats = reddit_df.filter(
        (col("period") == "pre_covid") & (col("subreddit") == subreddit_name)
    ).agg(
        avg("vader_sentiment").alias("mean"),
        stddev("vader_sentiment").alias("std"),
        count("vader_sentiment").alias("n")
    ).collect()[0]

    post_stats = reddit_df.filter(
        (col("period") == "post_covid") & (col("subreddit") == subreddit_name)
    ).agg(
        avg("vader_sentiment").alias("mean"),
        stddev("vader_sentiment").alias("std"),
        count("vader_sentiment").alias("n")
    ).collect()[0]

    print(f"r/{subreddit_name}")
    if pre_stats["n"] == 0:
        print(f"  Pre-COVID: no data (expected with test batch)")
        results.append((subreddit_name, "N/A", str(round(float(post_stats["mean"]),4)),
                        "N/A", "N/A", "N/A - no pre-COVID data"))
    else:
        m1,s1,n1 = pre_stats["mean"],pre_stats["std"],pre_stats["n"]
        m2,s2,n2 = post_stats["mean"],post_stats["std"],post_stats["n"]
        se = math.sqrt((s1**2/n1)+(s2**2/n2))
        t_stat = (m1-m2)/se
        sig = "YES" if abs(t_stat) > 1.96 else "NO"
        print(f"  Pre mean={m1:.4f}, Post mean={m2:.4f}, t={t_stat:.4f}, Significant: {sig}")
        results.append((subreddit_name, str(round(m1,4)), str(round(m2,4)),
                        str(round(t_stat,4)), "approx", sig))
    print()

print("results list ready, length:", len(results))

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

r/depression
  Pre-COVID: no data (expected with test batch)

r/anxiety
  Pre-COVID: no data (expected with test batch)

r/mentalhealth
  Pre-COVID: no data (expected with test batch)

results list ready, length: 3

## 4. Save All Results to S3
Save results locally for plotting.

In [10]:
monthly_stats.coalesce(1).write.mode("overwrite").csv(
    "s3://30113-final-project/results/plot_data/monthly_stats_csv/",
    header=True
)
print("monthly_stats saved.")

period_comparison.coalesce(1).write.mode("overwrite").csv(
    "s3://30113-final-project/results/plot_data/period_comparison_csv/",
    header=True
)
print("period_comparison saved.")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

monthly_stats saved.
period_comparison saved.

In [11]:
from pyspark.sql.types import StructType, StructField, StringType

schema = StructType([
    StructField("subreddit", StringType()),
    StructField("pre_mean", StringType()),
    StructField("post_mean", StringType()),
    StructField("t_stat", StringType()),
    StructField("p_value", StringType()),
    StructField("significant", StringType()),
])

results_df = spark.createDataFrame(results, schema=schema)
results_df.coalesce(1).write.mode("overwrite").csv(
    "s3://30113-final-project/results/plot_data/ttest_results_csv/",
    header=True
)
print("t-test results saved.")
results_df.show(truncate=False)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

t-test results saved.
+------------+--------+---------+------+-------+-----------------------+
|subreddit   |pre_mean|post_mean|t_stat|p_value|significant            |
+------------+--------+---------+------+-------+-----------------------+
|depression  |N/A     |0.0387   |N/A   |N/A    |N/A - no pre-COVID data|
|anxiety     |N/A     |0.0776   |N/A   |N/A    |N/A - no pre-COVID data|
|mentalhealth|N/A     |0.0771   |N/A   |N/A    |N/A - no pre-COVID data|
+------------+--------+---------+------+-------+-----------------------+

In [13]:
monthly_stats.write.mode("overwrite").parquet(
    "s3://30113-final-project/results/monthly_aggregates/"
)
period_comparison.write.mode("overwrite").parquet(
    "s3://30113-final-project/results/period_comparison/"
)
print("Parquet aggregates saved.")
print("\nAll done. Ready to download the CSVs from S3 and run 4_visualizations_local.py locally.")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Parquet aggregates saved.

All done. Ready to download the CSVs from S3 and run 4_visualizations_local.py locally.

## 5. Verify S3 Paths
Confirm all CSV files were saved correctly.

In [14]:
import boto3

s3 = boto3.client("s3")
prefixes = [
    "results/plot_data/monthly_stats_csv/",
    "results/plot_data/period_comparison_csv/",
    "results/plot_data/ttest_results_csv/"
]

for prefix in prefixes:
    response = s3.list_objects_v2(Bucket="30113-final-project", Prefix=prefix)
    files = [o["Key"] for o in response.get("Contents", [])]
    print(f"{prefix}:")
    for f in files:
        print(f"  {f}")
    if not files:
        print("  (empty — something went wrong above)")
    print()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

An error was encountered:
No module named 'boto3'
Traceback (most recent call last):
ModuleNotFoundError: No module named 'boto3'

